In [ ]:
!pip install pandas numpy scikit-learn fuzzywuzzy python-Levenshtein sentence-transformers tensorflow

In [ ]:
import tensorflow as tf
import pandas as pd
import json
import numpy as np
from google.colab import auth
from pydrive2.auth import GoogleAuth
from oauth2client.client import GoogleCredentials
from pydrive2.drive import GoogleDrive
from tqdm.notebook import tqdm
from datetime import datetime
from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from fuzzywuzzy import fuzz
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, GlobalMaxPooling1D
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [ ]:
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
gDrive = GoogleDrive(gauth)
gDrive.auth.settings['oauth_scope']
url = 'https://drive.google.com/file/d/1-uwjFTK6H5l4cVV8qjdBmoufAcuusB4R'
gDrive.CreateFile({'id':'1-uwjFTK6H5l4cVV8qjdBmoufAcuusB4R'}).GetContentFile('dataset_atividades.pkl')
atividades_df = pd.read_pickle('dataset_atividades.pkl')

In [ ]:
i  = 0
for grupo, sub_df in atividades_df.groupby(['sala','atividade']):
  i+=1
  if (i < 20):
    continue
  display(Markdown(f'## {grupo[0]} - {grupo[1]}'))
  vectorizer = TfidfVectorizer()
  tfidf_matrix = vectorizer.fit_transform(sub_df['corpo'])
  similarity_matrix = cosine_similarity(tfidf_matrix)
  similar = np.tril(similarity_matrix, k=-1)
  print(np.mean(similar[similar > 0]))
  print(similar)
  break

## ECOM09A 2025.1 IESTI - 4.2 - Treinamento de RNN

0.4850460891142085
[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.17502998 0.         0.         ... 0.         0.         0.        ]
 [0.64191994 0.13794339 0.         ... 0.         0.         0.        ]
 ...
 [0.61469412 0.13845504 0.60588655 ... 0.         0.         0.        ]
 [0.57608111 0.59683985 0.48337954 ... 0.47129403 0.         0.        ]
 [0.6365243  0.23732511 0.4196846  ... 0.48875508 0.53988945 0.        ]]


In [ ]:
i = 0
for grupo, sub_df in atividades_df.groupby(['sala','atividade']):
  i += 1
  if (i < 20):
    continue
  display(Markdown(f'## {grupo[0]} - {grupo[1]}'))
  docs_fuzz = sub_df['corpo'].tolist()
  n_docs_fuzz = len(docs_fuzz)
  fuzz_similarity_matrix = np.zeros((n_docs_fuzz, n_docs_fuzz))
  for i in range(n_docs_fuzz):
    for j in range(n_docs_fuzz):
      if i == j:
        fuzz_similarity_matrix[i, j] = 1.0
      elif i < j:
        score = fuzz.WRatio(str(docs_fuzz[i]), str(docs_fuzz[j])) / 100.0
        fuzz_similarity_matrix[i, j] = score
        fuzz_similarity_matrix[j, i] = score
  similar_values_fuzz = np.tril(fuzz_similarity_matrix, k=-1)
  positive_similarities_fuzz = similar_values_fuzz[similar_values_fuzz > 0]
  if positive_similarities_fuzz.size > 0:
    mean_similarity_fuzz = np.mean(positive_similarities_fuzz)
    print(f"Mean TheFuzz WRatio similarity: {mean_similarity_fuzz:.4f}")
  elif n_docs_fuzz >= 2 :
    print(f"Mean TheFuzz WRatio similarity: 0.0000 (all pairwise scores were 0)")
  else:
    print("No positive pairwise TheFuzz similarities found or no pairs to compare.")
  if n_docs_fuzz < 10 :
      print(np.round(similar_values_fuzz, 4))
  else:
      print(f'{n_docs_fuzz}x{n_docs_fuzz}')
      print(np.round(similar_values_fuzz[:10, :10], 4))
  break

SyntaxError: invalid syntax. Perhaps you forgot a comma? (<ipython-input-43-471bff45c1e1>, line 30)

In [ ]:
all_texts_corpus = atividades_df['corpo'].tolist()
keras_tokenizer = Tokenizer(num_words=15000, oov_token="<unk>")
keras_tokenizer.fit_on_texts(all_texts_corpus)

def create_lstm_embedder_model(max_words, embedding_dim, max_len, lstm_units):
    input_layer = Input(shape=(max_len,))
    embedding_layer = Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len)(input_layer)
    lstm_layer = LSTM(lstm_units, return_sequences=True)(embedding_layer)
    pooling_layer = GlobalMaxPooling1D()(lstm_layer)
    model = Model(inputs=input_layer, outputs=pooling_layer)
    return model

lstm_embedder_model = create_lstm_embedder_model(15000, 128, 500, 64)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
i = 0
for grupo, sub_df in atividades_df.groupby(['sala', 'atividade']):
    i += 1
    if (i < 20):
        continue
    display(Markdown(f'{grupo[0]} - {grupo[1]}'))
    docs_lstm = sub_df['corpo'].tolist()
    n_docs_lstm = len(docs_lstm)
    sequences = keras_tokenizer.texts_to_sequences(docs_lstm)
    padded_sequences = pad_sequences(sequences, maxlen=500)
    doc_embeddings = lstm_embedder_model.predict(padded_sequences, batch_size=32)
    lstm_similarity_matrix = cosine_similarity(doc_embeddings)
    similar_values_lstm = np.tril(lstm_similarity_matrix, k=-1)
    positive_similarities_lstm = similar_values_lstm[similar_values_lstm > 0]

    if positive_similarities_lstm.size > 0:
        mean_similarity_lstm = np.mean(positive_similarities_lstm)
        print(f"Mean LSTM Keras Cosine Similarity: {mean_similarity_lstm:.4f}")
    elif n_docs_lstm >= 2:
        print("Mean LSTM Keras Cosine Similarity: 0.0000 (todos os scores foram 0 ou negativos)")
    else:
        print("Nenhuma similaridade positiva encontrada.")

    if n_docs_lstm < 10:
        print(np.round(similar_values_lstm, 4))
    else:
        print(np.round(similar_values_lstm[:10, :10], 4))
    break

ECOM09A 2025.1 IESTI - 4.2 - Treinamento de RNN

2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step
Mean LSTM Keras Cosine Similarity: 0.9652
[[0.     0.     0.     0.     0.     0.     0.     0.     0.     0.    ]
 [0.9569 0.     0.     0.     0.     0.     0.     0.     0.     0.    ]
 [0.9713 0.9547 0.     0.     0.     0.     0.     0.     0.     0.    ]
 [0.9873 0.9537 0.9674 0.     0.     0.     0.     0.     0.     0.    ]
 [0.9742 0.9569 0.9601 0.9756 0.     0.     0.     0.     0.     0.    ]
 [0.9703 0.9532 0.9888 0.9707 0.9557 0.     0.     0.     0.     0.    ]
 [0.9473 0.9461 0.9746 0.9497 0.9496 0.9721 0.     0.     0.     0.    ]
 [0.9826 0.9542 0.9623 0.9833 0.9774 0.9563 0.9341 0.     0.     0.    ]
 [0.9418 0.929  0.9589 0.9383 0.9463 0.9503 0.9482 0.9436 0.     0.    ]
 [0.9876 0.957  0.9686 0.9807 0.9756 0.9681 0.9445 0.9824 0.9388 0.    ]]


In [ ]:
SENTENCE_TRANSFORMER_MODEL_NAME = 'all-MiniLM-L6-v2'
sentence_transformer_model = SentenceTransformer(SENTENCE_TRANSFORMER_MODEL_NAME, device='cpu')

In [ ]:
i = 0
for grupo, sub_df in atividades_df.groupby(['sala', 'atividade']):
    i += 1
    if (i < 20):
        continue
    display(Markdown(f'{grupo[0]} - {grupo[1]}'))
    docs_st = sub_df['corpo'].tolist()
    n_docs_st = len(docs_st)
    doc_embeddings = sentence_transformer_model.encode(docs_st, show_progress_bar=True, batch_size=32)
    st_similarity_matrix = cosine_similarity(doc_embeddings)
    similar_values_st = np.tril(st_similarity_matrix, k=-1)
    positive_similarities_st = similar_values_st[similar_values_st > 0]

    if positive_similarities_st.size > 0:
        mean_similarity_st = np.mean(positive_similarities_st)
        print(f"Mean Sentence Transformer Cosine Similarity: {mean_similarity_st:.4f}")
    elif n_docs_st >= 2:
        print("Mean Sentence Transformer Cosine Similarity: 0.0000 (todos os scores foram 0 ou negativos)")
    else:
        print("Nenhuma similaridade positiva encontrada.")

    if n_docs_st < 10:
        print(np.round(similar_values_st, 4))
    else:
        print(np.round(similar_values_st[:10, :10], 4))
    break

ECOM09A 2025.1 IESTI - 4.2 - Treinamento de RNN

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Mean Sentence Transformer Cosine Similarity: 0.5737
[[0.     0.     0.     0.     0.     0.     0.     0.     0.     0.    ]
 [0.5951 0.     0.     0.     0.     0.     0.     0.     0.     0.    ]
 [0.7294 0.6736 0.     0.     0.     0.     0.     0.     0.     0.    ]
 [0.7899 0.6561 0.8246 0.     0.     0.     0.     0.     0.     0.    ]
 [0.7202 0.7102 0.8069 0.7843 0.     0.     0.     0.     0.     0.    ]
 [0.6752 0.7301 0.8825 0.794  0.8344 0.     0.     0.     0.     0.    ]
 [0.79   0.6601 0.87   0.8815 0.8386 0.845  0.     0.     0.     0.    ]
 [0.4721 0.4749 0.4242 0.4862 0.4462 0.4481 0.4963 0.     0.     0.    ]
 [0.7552 0.6121 0.7432 0.8028 0.8019 0.7357 0.8449 0.545  0.     0.    ]
 [0.6347 0.6171 0.7407 0.6767 0.7725 0.8522 0.7027 0.4577 0.678  0.    ]]
